In [0]:
# Importação de funções
from pyspark.sql import functions as F

# Configuração de acesso ao Storage
spark.conf.set("fs.azure.account.key.stprojeto2zaca.dfs.core.windows.net", "7kEJbeQdNyThcjCw9jHUrHBG4nCsf67Oeltzaw/5pW7ptO3bhOM2s5cZ7comiXjgJHd04GMim77r+ASt/wB8Ow==")

# 1. Leitura dos dados da camada Silver
path_silver = "abfss://dados@stprojeto2zaca.dfs.core.windows.net/02-silver/tabela_corridas"
df_silver = spark.read.format("delta").load(path_silver)

# 2. Transformação: Agregação por Data (KPIs de Negócio)
# Usando 'to_date' para garantir que o agrupamento seja por dia
df_gold = df_silver.groupBy(F.to_date("pickup_datetime").alias("data_corrida")) \
    .agg(
        F.count("key").alias("total_corridas"),
        F.sum("fare_amount").alias("faturamento_total"),
        F.avg("fare_amount").alias("ticket_medio")
    ) \
    .orderBy("data_corrida")

# 3. Escrita na camada Gold
path_gold = "abfss://dados@stprojeto2zaca.dfs.core.windows.net/03-gold/kpi_faturamento_diario"
df_gold.write.format("delta").mode("overwrite").save(path_gold)

In [0]:
df_silver.printSchema()